# Speculative Decoding Evaluation: Llama-3-8B Target & Llama-3.2-1B Draft (Hindi FLORES+)

An empirical and simulated investigation of speculative decoding for English-to-Hindi machine translation, scaling the target model from 3B to 8B across dual Tesla T4 GPUs, evaluating static windows, adaptive entropy stopping, and linguistic boundary policies.

---

## Executive Summary of Experiments

This notebook conducts a complete end-to-end study of speculative decoding across three primary phases:

### Part A: Empirical Target Generation, Draft Traces & Hardware Profiling (GPU)
1. **Target 8B Translation Generation (FLORES+)**:
   - Deploys `Meta-Llama-3-8B-Instruct` (fp16) sharded across 2x Tesla T4 GPUs via `device_map="auto"` (8 GiB on GPU 0, 10 GiB on GPU 1).
   - Translates the parallel English-to-Hindi `devtest` split (1,012 sentences) using batched greedy generation with fault-tolerant JSONL streaming.
2. **Draft 1B Teacher-Forced Pass & Uncertainty Tracing**:
   - Executes `Llama-3.2-1B-Instruct` on GPU 0 to evaluate `[prompt + target_output]` prefixes in a single forward pass per sentence.
   - Validates exact vocabulary and subword tokenization parity with the 8B target.
   - Records token-level argmax agreement, target probabilities, and per-token Shannon entropy ($H$ in nats) into `draft_traces.json`.
3. **Hardware Latency Benchmarking & Cost Ratio Modeling**:
   - Microbenchmarks single-token auto-regressive decode steps with KV-caching for draft ($t_{\text{draft}}$) and target ($t_{\text{target}}$).
   - Computes empirical cost ratio $c = t_{\text{draft}} / t_{\text{target}}$ (accounting for inter-GPU communication across the sharded target).
   - Measures multi-token parallel verification latencies for candidate lengths $\gamma+1 \in [2 \dots 9]$ to establish realistic verification cost scaling.

### Part B: Offline Speculative Decoding Simulation & Policy Optimization (CPU)
4. **Discrete-Event Trace Simulator & Unit Testing**:
   - Simulates speculative decoding verification rounds offline from collected traces without GPU overhead.
   - Compares empirical performance against the analytical closed-form i.i.d. speedup formula $f = \frac{1 - \alpha^{\gamma+1}}{(1 - \alpha)(\gamma c + 1)}$.
   - Validates simulation correctness across boundary edge cases and dataset invariants.
5. **Fixed-$k$ Window Sweep ($k = 2 \dots 8$)**:
   - Evaluates constant draft lengths, quantifying acceptance rate decay, token yield per verification, and net speedup.
   - Measures performance gap between simulated speedup and the theoretical i.i.d. formula caused by correlated subword errors.
6. **Adaptive Entropy-Guided Early Stopping (AdaEDL / SVIP Rule)**:
   - Evaluates dynamic draft halting based on draft token entropy: stop when $1 - \sqrt{\beta H} < \lambda$.
   - Splits traces 50/50 into dev and held-out test partitions (`SEED=0`).
   - Conducts a 2D hyperparameter grid sweep over entropy scale $\beta \in \{0.5, 1.0, 2.0\}$ and confidence thresholds $\lambda \in [-1.5, 0.95]$ on dev data.
7. **Theoretical Oracle Upper Bounds**:
   - Evaluates an ideal Oracle that only drafts tokens accepted by the target, establishing the maximum achievable speedup ceiling at cap $\gamma_{\max}=8$ and uncapped $\gamma_{\max}=32$.
8. **Held-Out Test Evaluation & Comparative Summary**:
   - Benchmarks dev-selected optimal configurations on held-out test data.
   - Exports comparative results to CSV tables and visualizes speedup gains across policies.

### Extended Experiments: Linguistic & Boundary-Aware Speculative Decoding
9. **Linguistic Feature Extraction & Boundary Tagging**:
   - Decodes drafted subwords into Hindi text to identify orthographic and lexical boundaries.
   - Evaluates a dictionary lookup rule (`MORPH_DICT`) on high-frequency Hindi grammatical morphemes.
   - Evaluates a Devanagari Akshara regex (`AKSHARA_REGEX`) to enforce syllabic completeness and avoid splitting combining signs (halants/matras).
10. **Linguistic Stopping Policies**:
    - **Word Boundary Policy**: Halts drafting when whitespace, punctuation, or word-final boundaries are reached.
    - **Morpheme Lookup Policy**: Halts drafting when accumulated subwords complete a known grammatical morpheme.
    - **Hybrid Akshara Policy**: Halts drafting at valid orthographic akshara or lexical boundaries.
11. **Entropy vs. Boundary Redundancy Analysis**:
    - Calculates point-biserial correlation ($r_{\text{pb}}$) between token entropy and word boundaries to determine whether linguistic rules provide orthogonal stopping signals or mirror natural model uncertainty.

---

## Kaggle Environment & Execution Guide
1. **Accelerator**: **GPU T4 x2** (Settings > Accelerator > GPU T4 x2).
2. **Internet**: **On** (required for Hugging Face Hub model/dataset download).
3. **Secrets**: Add `HF_TOKEN` in *Add-ons > Secrets* with authorized access to Meta-Llama-3 models.
4. **Fast Smoke Testing**: Set `N = 20` in the hyperparameter cell to test the full pipeline before executing on all 1,012 sentences (`N = None`).

### Environment Setup & Dependency Installation

Configures PyTorch memory management and installs required libraries:
- **CUDA Allocator**: Sets `PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"` before PyTorch is imported to reduce memory fragmentation across dual GPUs.
- **Dependencies**: Installs current versions of `transformers`, `accelerate`, `datasets`, `huggingface_hub`, and `pandas`.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set before torch is imported
!pip -q install -U transformers accelerate datasets huggingface_hub pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 29.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you hav

### Hugging Face Hub Authentication

Authenticates access to gated model weights and datasets:
- **Kaggle Secret**: Attempts automated login using the stored `HF_TOKEN` secret via `UserSecretsClient`.
- **Interactive Fallback**: Prompts for device/token login if the secret is unavailable.
- **Scope**: Required for `Meta-Llama-3-8B-Instruct`, `Llama-3.2-1B-Instruct`, and `openlanguagedata/flores_plus`.

In [2]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in with Kaggle secret HF_TOKEN")
except Exception as e:
    print(f"Kaggle secret HF_TOKEN not available ({e}); paste your token below")
    login()

Kaggle secret HF_TOKEN not available (Connection error trying to communicate with service.); paste your token below


# Part A: Day 2 (target outputs, draft traces, cost ratio)

### Hardware Verification, Hyperparameters & Tokenizer Setup

Configures experiment settings, verifies dual-GPU hardware, and loads the target tokenizer:
- **Experiment Settings**: Defines target (`Meta-Llama-3-8B-Instruct`) and draft (`Llama-3.2-1B-Instruct`) models in `float16`, batch size (16), generation budget (`MAX_NEW = 256`), and output directory.
- **Memory Sharding Config**: Sets `MAX_MEMORY` ({GPU 0: 8 GiB, GPU 1: 10 GiB}) so the 8B model spans both T4 GPUs while reserving headroom on GPU 0 for the 1B draft model and KV-cache.
- **GPU Diagnostics**: Asserts $\ge 2$ GPUs are available and defines CUDA synchronization (`sync_all`) and memory tracking (`gpu_report`) utilities.
- **Tokenizer Initialization**: Loads Llama-3-8B tokenizer with left-padding enabled and maps pad token to EOS for batched autoregressive generation.

In [3]:
import gc, json, time, re
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

TARGET = "meta-llama/Meta-Llama-3-8B-Instruct"   # alt: "meta-llama/Llama-3.1-8B-Instruct" (officially supports Hindi; same family as the 3.2-1B draft)
DRAFT  = "meta-llama/Llama-3.2-1B-Instruct"
DTYPE  = torch.float16          # T4 has no real bf16
N      = None                   # None = all 1012 sentences; use 20 for a smoke test
BATCH, MAX_NEW = 16, 256        # lower BATCH to 8 if generation runs out of memory
dev = "cuda:0"                  # inputs enter the (sharded) target here; the draft also lives here
MAX_MEMORY = {0: "8GiB", 1: "10GiB"}   # cap for the 8B target per GPU (GPU 0 keeps room for the 1B draft + KV cache)
OUT = Path("/kaggle/working/day2_8b"); OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.device_count() >= 2, "Need 2 GPUs: Settings > Accelerator > GPU T4 x2"
for i in range(torch.cuda.device_count()):
    print(f"GPU{i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GiB")

def sync_all():
    for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)

def gpu_report(tag=""):
    print(tag, " | ".join(f"GPU{i}: {torch.cuda.memory_allocated(i)/2**30:.1f} GiB allocated" for i in range(torch.cuda.device_count())))

tok = AutoTokenizer.from_pretrained(TARGET)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token

AssertionError: Need 2 GPUs: Settings > Accelerator > GPU T4 x2

### Load Evaluation Dataset (FLORES+ English-to-Hindi)

Prepares parallel sentences for translation evaluation:
- **Data Ingestion**: Loads the `devtest` split of `openlanguagedata/flores_plus` for English (`eng_Latn`) and Hindi (`hin_Deva`).
- **Alignment**: Aligns English and Hindi translations by sentence ID into parallel `(id, src, ref)` records.
- **Smoke Testing**: Slices the dataset to `N` items if specified (e.g., $N=20$ for test runs, or all 1,012 sentences).

In [ ]:
from datasets import load_dataset
# If gated access isn't approved yet, load the GitHub-release text files instead.
en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
hi = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split="devtest")
hi_by_id = {r["id"]: r["text"] for r in hi}
rows = [(r["id"], r["text"], hi_by_id[r["id"]]) for r in en]
if N: rows = rows[:N]
ids, src, ref = zip(*rows)
print(len(rows), "sentences\n", src[0], "\n", ref[0])

## 1. Target (8B) Greedy Translations

Loads the sharded 8B model and generates greedy baseline Hindi translations:
- **Prompt Formatting**: Wraps English source sentences into Llama-3 instruction chat templates (`build_prompt`).
- **Model Sharding**: Loads `Meta-Llama-3-8B-Instruct` across both T4 GPUs via `device_map="auto"` conforming to `MAX_MEMORY` caps.
- **Stopping Criteria**: Identifies standard EOS and `<|eot_id|>` tokens as termination conditions.
- **Batched Generation**: Executes greedy generation (`do_sample=False`) in batches of 16.
- **Streaming Checkpoints**: Appends generated translations (`out_ids`, `out_text`) to `target_outputs.jsonl` after every batch, enabling automatic resumption if interrupted.

In [ ]:
def build_prompt(s):
    msgs = [{"role": "user", "content":
             f"Translate the following English sentence into Hindi. Output only the Hindi translation.\n\n{s}"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

prompts = [build_prompt(s) for s in src]

target = AutoModelForCausalLM.from_pretrained(
    TARGET, torch_dtype=DTYPE, device_map="auto", max_memory=MAX_MEMORY, low_cpu_mem_usage=True,
).eval()
print("modules per device:", dict(Counter(target.hf_device_map.values())))
gpu_report("after loading target:")

# stop ids: whatever the generation config says, plus <|eot_id|> (Llama-3 instruct end-of-turn)
eos = target.generation_config.eos_token_id
eos_ids = set(eos if isinstance(eos, list) else [eos])
eos_ids.add(tok.convert_tokens_to_ids("<|eot_id|>"))

OUT_JSONL = OUT / "target_outputs.jsonl"
records = []
if OUT_JSONL.exists():
    records = [json.loads(l) for l in open(OUT_JSONL)]
    if [r["id"] for r in records] != list(ids[:len(records)]):   # stale file from another run / another N
        records = []
    print(f"resuming: {len(records)} / {len(rows)} sentences already done")

with open(OUT_JSONL, "a" if records else "w") as f:
    for i in tqdm(range(len(records), len(prompts), BATCH)):
        enc = tok(prompts[i:i+BATCH], return_tensors="pt", padding=True, add_special_tokens=False).to(dev)
        with torch.no_grad():
            out = target.generate(**enc, do_sample=False, temperature=None, top_p=None,
                                  max_new_tokens=MAX_NEW, eos_token_id=list(eos_ids),
                                  pad_token_id=tok.pad_token_id)
        gen = out[:, enc.input_ids.shape[1]:].tolist()
        for j, g in enumerate(gen):
            cut = next((p+1 for p, t in enumerate(g) if t in eos_ids), len(g))   # keep the eos token
            g = g[:cut]
            pid = enc.input_ids[j][enc.attention_mask[j].bool()].tolist()        # prompt without padding
            rec = dict(id=ids[i+j], src=src[i+j], ref=ref[i+j], prompt_ids=pid,
                       out_ids=g, out_text=tok.decode(g, skip_special_tokens=True).strip())
            records.append(rec); f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()

print(f"{len(records)} target outputs saved to {OUT_JSONL}")
for r in records[:3]: print(r["src"], "\n ->", r["out_text"], "\n")
n_hit = sum(len(r["out_ids"]) >= MAX_NEW for r in records)
print(f"{n_hit} outputs hit MAX_NEW={MAX_NEW} without finishing")

## 2. Draft (1B) Teacher-Forced Pass: Argmax & Entropy Traces

Runs the draft model over target outputs to record prediction agreement and uncertainty:
- **Model Loading**: Loads `Llama-3.2-1B-Instruct` onto GPU 0 in `float16` alongside the sharded target model.
- **Tokenizer Compatibility**: Validates that draft and target models share identical vocabulary size and exact subword tokenization.
- **Teacher-Forced Logits**: Evaluates `[prompt + target_output]` through the draft model in single forward passes to obtain predictions at each position $t$.
- **Trace Extraction**: Computes draft argmax tokens, Shannon entropy $H$ (in nats), probability assigned to target tokens (`p_target`), and binary match indicators (`match`).
- **Trace Serialization**: Saves trace records to `draft_traces.json` and outputs aggregate agreement rate and entropy breakdown (matching vs. mismatching tokens).

In [ ]:
gc.collect(); torch.cuda.empty_cache()
draft = AutoModelForCausalLM.from_pretrained(DRAFT, torch_dtype=DTYPE).to(dev).eval()
gpu_report("after loading draft:")

# draft and target must share a vocabulary, otherwise the whole experiment is meaningless
tok_d = AutoTokenizer.from_pretrained(DRAFT)
assert draft.config.vocab_size == target.config.vocab_size, "vocab sizes differ"
probe = (records[0]["out_text"] + " नमस्ते दुनिया. Hello world!")
assert tok_d(probe, add_special_tokens=False).input_ids == tok(probe, add_special_tokens=False).input_ids, "tokenizers disagree"
print("draft/target tokenizers match")

traces = []
for r in tqdm(records):
    p, o = r["prompt_ids"], r["out_ids"]
    x = torch.tensor([p + o], device=dev)
    with torch.no_grad():
        lg = draft(x).logits[0, len(p)-1:-1].float()   # position t predicts output token t
    lp  = torch.log_softmax(lg, -1)
    ent = -(lp.exp() * lp).sum(-1)                      # entropy in nats
    am  = lp.argmax(-1)
    tg  = torch.tensor(o, device=dev)
    traces.append(dict(id=r["id"], target_ids=o, draft_argmax=am.tolist(), entropy=ent.tolist(),
                       p_target=lp.gather(1, tg[:, None]).squeeze(1).exp().tolist(),
                       match=(am == tg).tolist()))

json.dump(traces, open(OUT/"draft_traces.json", "w"))
m = np.concatenate([t["match"] for t in traces]); e = np.concatenate([t["entropy"] for t in traces])
print(f"positions: {len(m)} | token-level draft/target agreement: {m.mean():.3f} | mean entropy: {e.mean():.3f} nats")
print(f"mean entropy when match: {e[m].mean():.3f} | when mismatch: {e[~m].mean():.3f}")

## 3. Forward-Pass Benchmarking & Cost Ratio ($c$)

Empirically benchmarks decode and verification latencies to compute the relative cost ratio:
- **Timing Harness (`time_fwd`)**: Measures forward latency at batch size 1 with KV-caching active, synchronizing CUDA across warm-up and timed iterations.
- **Cost Ratio ($c$)**: Measures single-token decode latency for draft ($t_{\text{draft}}$) and target ($t_{\text{target}}$) to compute $c = t_{\text{draft}} / t_{\text{target}}$ (incorporating inter-GPU communication across the sharded target).
- **Verification Latencies**: Measures target verification latency for candidate lengths $\gamma+1$ tokens ($\gamma \in \{1 \dots 8\}$).
- **Export**: Writes latency profiles, hardware configurations, and cost ratio $c$ to `timing.json`.

In [ ]:
@torch.no_grad()
def time_fwd(model, q=1, ctx=150, steps=60, reps=3):
    ts = []
    for _ in range(reps):
        x = torch.randint(1000, 20000, (1, ctx), device=dev)
        o = model(x, use_cache=True); pkv = o.past_key_values
        for s in range(steps + 10):
            inp = torch.randint(1000, 20000, (1, q), device=dev)
            sync_all(); t = time.perf_counter()
            o = model(inp, past_key_values=pkv, use_cache=True)
            sync_all(); dt = time.perf_counter() - t
            pkv = o.past_key_values
            if s >= 10: ts.append(dt)
    return float(np.median(ts))

t_d = time_fwd(draft, 1)
t_t = time_fwd(target, 1)
print(f"draft  (1B) decode step: {t_d*1000:.2f} ms")
print(f"target (8B) decode step: {t_t*1000:.2f} ms")
print(f"c = t_draft / t_target = {t_d/t_t:.3f}")

# Verification cost for gamma+1 tokens (should be close to one decode step if memory-bound).
# gamma=1 is included too, since dynamic policies can draft a single token.
ver = {g: time_fwd(target, g+1) for g in range(1, 9)}
print({g: f"{v*1000:.1f} ms" for g, v in ver.items()})

json.dump(dict(t_draft=t_d, t_target=t_t, c=t_d/t_t, t_target_verify={str(g): v for g, v in ver.items()},
               setup=dict(target=TARGET, draft=DRAFT, dtype=str(DTYPE), max_memory=MAX_MEMORY,
                          gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])),
          open(OUT/"timing.json", "w"), indent=1)

### GPU Resource Deallocation & Transition to CPU

Frees GPU memory before running offline CPU trace simulations:
- **VRAM Deallocation**: Explicitly deletes target and draft model objects and runs garbage collection and CUDA cache clearing.
- **Verification**: Confirms GPU memory deallocation and verifies that all Part A artifacts (`draft_traces.json`, `target_outputs.jsonl`, `timing.json`) are safely written to disk.

In [ ]:
# Free the GPUs; Part B is pure CPU.
del target, draft
gc.collect(); torch.cuda.empty_cache()
gpu_report("after cleanup:")
print(sorted(p.name for p in OUT.iterdir()))

# Part B: Day 3 (trace simulator, fixed-k baseline, entropy policy, oracle)

Simulates speculative decoding *offline* from the Part A traces (no GPU needed). If you start a fresh session for this part, add the saved `day2_8b` folder as a Kaggle dataset and point `DATA_DIR` at `/kaggle/input/<your-dataset>/`.

**Why a trace simulator is exact for greedy decoding:** within a draft window, the draft's tokens are conditioned on its own previous drafts. Those coincide with the target's tokens up to the first mismatch, and everything after the first mismatch is rejected anyway. So the number accepted = number of leading matches in the window.

**Cost model:** one round with `g` drafted tokens costs `g*c + 1` (target forward = 1, draft forward = `c`) and yields `accepted + 1` tokens (the +1 is the correction or bonus token from the target). Plain decoding costs 1 per token, so `speed-up = tokens emitted / total cost`.

### Part B Setup: Trace Ingestion & Verification Cost Model

Prepares the offline simulation environment and deserializes recorded traces:
- **Paths & Config**: Configures working directories, seeds (`SEED=0`), draft evaluation range $K = [2 \dots 8]$, and dynamic policy cap $\gamma_{\max}=8$.
- **Trace Ingestion (`load_traces`)**: Parses `draft_traces.json` into NumPy arrays with consistency checks across target tokens, draft tokens, and entropy values.
- **Cost Interpolation (`VERIFY`)**: Reads cost ratio $C$ from `timing.json` and builds a linear interpolator for verification latency as a function of draft length $\gamma$.
- **Dataset Statistics**: Reports total sentence count, target token count, and aggregate per-token draft/target agreement rate.

In [ ]:
import json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR    = Path("/kaggle/working/day2_8b")      # folder with draft_traces.json + timing.json from Part A
RES_DIR     = DATA_DIR / "day3"; RES_DIR.mkdir(parents=True, exist_ok=True)
TRACE_PATH  = DATA_DIR / "draft_traces.json"
TIMING_PATH = DATA_DIR / "timing.json"
K_RANGE     = range(2, 9)                          # fixed windows k = 2..8
GAMMA_MAX   = 8                                    # draft cap for dynamic policies (same as max k, for a fair comparison)
SEED        = 0
random.seed(SEED); np.random.seed(SEED)

def load_traces(path):
    traces = []
    for r in json.load(open(path)):
        tgt = np.asarray(r["target_ids"]); drf = np.asarray(r["draft_argmax"])
        H = np.asarray(r["entropy"], dtype=float)
        assert len(tgt) == len(drf) == len(H), f"length mismatch in sentence {r.get('id')}"
        match = (tgt == drf)
        assert np.array_equal(match, np.asarray(r["match"], bool)), f"match mismatch in sentence {r.get('id')}"
        traces.append(dict(id=r.get("id"), n=len(tgt), match=match, H=H, target_ids=tgt,
                           p_target=np.asarray(r.get("p_target", []), float), tokens=None))
    return traces

assert TRACE_PATH.exists() and TIMING_PATH.exists(), f"missing Part A outputs in {DATA_DIR}"
traces = load_traces(TRACE_PATH)

# Cost model. c = draft step / target step. Optionally use the measured verification cost for gamma+1 tokens too.
T = json.load(open(TIMING_PATH)); C = T["c"]
gs = sorted(int(g) for g in T["t_target_verify"])
vs = [max(1.0, T["t_target_verify"][str(g)] / T["t_target"]) for g in gs]   # clip: verify can't beat one decode step (dips are timing noise)
VERIFY = lambda g: 1.0 if g == 0 else float(np.interp(g, gs, vs))
print(f"c = {C:.3f} (measured) | verify cost / decode step for gamma={gs[0]}..{gs[-1]}: {[round(v, 3) for v in vs]}")

n_tok = sum(s["n"] for s in traces)
print(f"{len(traces)} sentences, {n_tok} target tokens, per-token draft/target match rate = {np.concatenate([s['match'] for s in traces]).mean():.3f}")

## Speculative Decoding Trace Simulator & Drafting Policies

Defines the discrete-event trace simulator and candidate stopping policies:
- **Simulator Engine (`simulate`)**: Emulates speculative verification rounds offline by querying `policy(s, p, g) -> bool` before each draft step, checking accepted tokens $a$, adding 1 bonus/correction token from the target, and accumulating costs ($g \cdot c + \text{verify}(g)$).
- **Metric Summarizer (`summarize`)**: Computes acceptance rate, empirical per-token accept probability $\hat{\alpha}$, tokens emitted per verification round, average draft length, and net speedup.
- **Analytical Speedup (`formula_speedup`)**: Calculates closed-form speedup under the i.i.d. assumption: $\frac{1 - \alpha^{\gamma+1}}{(1 - \alpha)(\gamma c + 1)}$.
- **Drafting Policies**:
  - `FixedK`: Constant draft window of length $k$.
  - `EntropyPolicy`: AdaEDL/SVIP early-stopping rule triggered when $1 - \sqrt{\beta H} < \lambda$.
  - `Oracle`: Theoretical upper bound; drafts exactly the consecutive matching prefix without wasting draft compute.
  - `NoDraft`: Standard non-speculative autoregressive baseline (speedup = 1.0).

In [ ]:
def simulate(traces, policy, gamma_max, c, verify=None):
    rounds = drafted = accepted = rejections = emitted = 0
    cost = 0.0
    draft_lens = []
    for s in traces:
        n, match = s["n"], s["match"]
        p = 0
        while p < n:
            r = n - p                              # tokens still to produce
            cap = min(gamma_max, r)
            g = 0
            while g < cap and not policy(s, p, g):  # draft one more token
                g += 1
            a = 0                                  # leading matches = accepted
            while a < g and match[p + a]:
                a += 1
            gained = min(a + 1, r)                 # + correction/bonus token from target
            rounds += 1; drafted += g; accepted += a
            rejections += int(a < g); emitted += gained
            cost += g * c + (verify(g) if verify else 1.0)
            draft_lens.append(g)
            p += gained
    return dict(rounds=rounds, drafted=drafted, accepted=accepted, rejections=rejections,
                emitted=emitted, cost=cost, draft_lens=draft_lens)

def summarize(res):
    return dict(
        acceptance_rate=res["accepted"] / max(res["drafted"], 1),                         # accepted / drafted
        alpha_hat=res["accepted"] / max(res["accepted"] + res["rejections"], 1),          # per-token accept prob (geometric MLE)
        tokens_per_verif=res["emitted"] / res["rounds"],
        mean_draft_len=res["drafted"] / res["rounds"],
        speedup=res["emitted"] / res["cost"],
    )

def formula_speedup(alpha, gamma, c):
    # f = (1 - a^(g+1)) / ((1 - a)(g c + 1)),  i.i.d. acceptance assumption
    exp_tokens = (gamma + 1) if alpha >= 1 else (1 - alpha ** (gamma + 1)) / (1 - alpha)
    return exp_tokens / (gamma * c + 1)

# ---- policies ----
class FixedK:
    def __init__(self, k): self.k = k
    def __call__(self, s, p, g): return g >= self.k

class EntropyPolicy:
    # AdaEDL/SVIP-style: after drafting a token, stop if 1 - sqrt(beta * H) < lam.
    # NOTE: beta is the entropy scale hyperparameter (called gamma in SVIP), NOT the draft length.
    def __init__(self, lam, beta=1.0): self.lam, self.beta = lam, beta
    def __call__(self, s, p, g):
        if g == 0: return False                    # always draft at least one token
        H = max(s["H"][p + g - 1], 0.0)            # entropy of the last drafted token
        return 1.0 - math.sqrt(self.beta * H) < self.lam

class Oracle:
    # Upper bound: knows the future; drafts exactly the leading run of matches, never wastes a draft token.
    def __call__(self, s, p, g): return not s["match"][p + g]

class NoDraft:
    def __call__(self, s, p, g): return True

### Simulator Sanity Checks & Invariant Validation

Verifies simulation correctness across deterministic edge cases and dataset invariants:
- **Edge-Case Validation**: Tests perfect agreement (emits $k+1$ tokens per round), complete disagreement (0 acceptances, 1 token emitted per round), and `NoDraft` (exact $1.0\times$ speedup).
- **Oracle Check**: Confirms `Oracle` incurs 0 rejections on full traces.
- **Token Invariant**: Asserts that every policy emits exactly the total number of target tokens ($n_{\text{tok}}$).

In [ ]:
# ---- sanity checks on tiny hand-made traces ----
def _t(match):
    m = np.array(match, bool); return dict(id=0, n=len(m), match=m, H=np.zeros(len(m)), tokens=None)

# all match, k=4, c=0: 10 tokens -> rounds emit 5,5 => 2 rounds
r = simulate([_t([1]*10)], FixedK(4), 4, 0.0); assert r["rounds"] == 2 and r["emitted"] == 10, r
# nothing matches: every round drafts k, accepts 0, emits 1
r = simulate([_t([0]*6)], FixedK(3), 3, 0.0); assert r["rounds"] == 6 and r["accepted"] == 0 and r["emitted"] == 6, r
# no drafting == speed-up 1
r = simulate([_t([1]*7)], NoDraft(), 8, 0.3); assert abs(summarize(r)["speedup"] - 1) < 1e-9
# oracle never rejects
r = simulate(traces, Oracle(), GAMMA_MAX, C); assert r["rejections"] == 0 and r["emitted"] == n_tok
# every policy emits exactly the number of target tokens
for pol in [FixedK(5), EntropyPolicy(0.5)]:
    assert simulate(traces, pol, GAMMA_MAX, C)["emitted"] == n_tok
print("sanity checks passed")

## 1. Fixed-k Baseline Evaluation (k = 2..8)

Sweeps static draft lengths $k \in \{2 \dots 8\}$ to benchmark fixed-window speculation:
- **Simulation**: Evaluates each $k$ using constant verification cost ($1.0$) and empirical verification latency scaling (`VERIFY`).
- **Theoretical Comparison**: Calculates analytical i.i.d. formula speedup using empirical $\hat{\alpha}$ to highlight performance gaps from non-i.i.d. acceptance.
- **Output**: Generates `fixed_df` comparing acceptance rates, per-verification token yield, and net speedups.

In [ ]:
rows = []
for k in K_RANGE:
    res = simulate(traces, FixedK(k), k, C); m = summarize(res)
    m2 = summarize(simulate(traces, FixedK(k), k, C, VERIFY))
    rows.append(dict(k=k, **m, formula_speedup=formula_speedup(m["alpha_hat"], k, C), speedup_measured_verify=m2["speedup"]))
fixed_df = pd.DataFrame(rows).set_index("k")
print(f"c = {C:.3f}")
fixed_df.round(3)

### Visualizing Fixed-k Performance Metrics

Plots diagnostic trends across draft window lengths $k$:
- **Subplots**:
  1. Acceptance rate vs. empirical per-token acceptance probability $\hat{\alpha}$.
  2. Accepted tokens per verification round (including bonus token).
  3. Simulated speedup vs. i.i.d. formula speedup and measured verification speedup.
- **Optimum Selection**: Computes and prints `best_k` achieving maximum empirical speedup.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(fixed_df.index, fixed_df.acceptance_rate, "o-", label="accepted / drafted")
ax[0].plot(fixed_df.index, fixed_df.alpha_hat, "s--", label="alpha_hat (per-token)")
ax[0].set(xlabel="k", ylabel="rate", title="Acceptance vs k"); ax[0].legend()
ax[1].plot(fixed_df.index, fixed_df.tokens_per_verif, "o-")
ax[1].set(xlabel="k", ylabel="tokens / verification", title="Accepted tokens per verification (incl. +1)")
ax[2].plot(fixed_df.index, fixed_df.speedup, "o-", label="simulated")
ax[2].plot(fixed_df.index, fixed_df.formula_speedup, "s--", label="formula (iid)")
ax[2].plot(fixed_df.index, fixed_df.speedup_measured_verify, "^:", label="simulated, measured verify cost")
ax[2].axhline(1, color="gray", lw=0.8); ax[2].set(xlabel="k", ylabel="speed-up", title="Speed-up vs k"); ax[2].legend()
plt.tight_layout(); plt.show()
best_k = int(fixed_df.speedup.idxmax()); print("best fixed k:", best_k, f"-> {fixed_df.speedup.max():.3f}x")

## 2. Entropy Policy Tuning (AdaEDL/SVIP Rule)

Optimizes early-exit threshold $\lambda$ and entropy scale $eta$ on a dev set:
- **Dev/Test Split**: Shuffles and splits traces 50/50 into `dev_set` and held-out `test_set` using `SEED=0`.
- **Grid Sweep**: Evaluates 40 threshold values $\lambda \in [-1.5, 0.95]$ across entropy multipliers $\beta \in \{0.5, 1.0, 2.0\}$ on `dev_set`.
- **Optimization & Plotting**: Identifies optimal $(\beta^*, \lambda^*)$ maximizing dev speedup and plots performance curves against the `best_k` baseline.

In [ ]:
idx = np.random.default_rng(SEED).permutation(len(traces))
dev_set  = [traces[i] for i in idx[: len(idx)//2]]
test_set = [traces[i] for i in idx[len(idx)//2 :]]

LAMS  = np.round(np.linspace(-1.5, 0.95, 40), 3)
BETAS = [0.5, 1.0, 2.0]
sweep = []
for beta in BETAS:
    for lam in LAMS:
        m = summarize(simulate(dev_set, EntropyPolicy(lam, beta), GAMMA_MAX, C))
        sweep.append(dict(beta=beta, lam=lam, **m))
sweep_df = pd.DataFrame(sweep)
best = sweep_df.loc[sweep_df.speedup.idxmax()]
print("best on dev:\n", best.round(3).to_string())

plt.figure(figsize=(6, 4))
for beta, g in sweep_df.groupby("beta"): plt.plot(g.lam, g.speedup, "o-", ms=3, label=f"beta={beta}")
plt.axhline(summarize(simulate(dev_set, FixedK(best_k), best_k, C))["speedup"], color="gray", ls="--", label=f"fixed k={best_k}")
plt.xlabel("lambda"); plt.ylabel("speed-up (dev)"); plt.title("Entropy policy: lambda sweep"); plt.legend(); plt.show()

## 3. Oracle Upper Bound Benchmark

Evaluates the theoretical maximum speedup attainable without wasted draft compute:
- **Capped Oracle ($\gamma_{\max}=8$ hold-out)**: Establishes performance ceiling when restricted to the same maximum draft length as heuristic policies.
- **Uncapped Oracle ($\gamma_{\max}=32$ hold-out)**: Determines the theoretical speedup headroom if the draft window was substantially extended.

In [ ]:
oracle_res = simulate(test_set, Oracle(), GAMMA_MAX, C)
oracle_unc = simulate(test_set, Oracle(), 32, C)
print("oracle (cap 8): ", {k: round(v, 3) for k, v in summarize(oracle_res).items()})
print("oracle (cap 32):", {k: round(v, 3) for k, v in summarize(oracle_unc).items()})

## 4. Final Evaluation & Summary on Test Split

Benchmarks dev-selected policies on held-out test data and exports results:
- **Policy Comparison**: Evaluates `no speculation`, dev-optimal `fixed k`, dev-tuned `entropy policy`, and `oracle` on `test_set`.
- **Table Generation**: Builds comparative summary table (`summary_df`) including both standard and measured-verification speedups.
- **Artifact Export**: Saves test summary and sweep tables to CSV files in `RES_DIR` (`day3_summary_test_8b.csv`, `day3_fixed_k_full_8b.csv`, `day3_entropy_sweep_dev_8b.csv`).
- **Bar Chart**: Visualizes final test speedups across all candidate policies.

In [ ]:
best_k_dev = max(K_RANGE, key=lambda k: summarize(simulate(dev_set, FixedK(k), k, C))["speedup"])
policies = {
    "no speculation":            (NoDraft(), GAMMA_MAX),
    f"fixed k={best_k_dev} (dev-best)": (FixedK(best_k_dev), best_k_dev),
    f"entropy (lam={best.lam:.2f}, beta={best.beta})": (EntropyPolicy(best.lam, best.beta), GAMMA_MAX),
    "oracle (cap 8)":            (Oracle(), GAMMA_MAX),
}
rows = {}
for name, (pol, cap) in policies.items():
    res = simulate(test_set, pol, cap, C); rows[name] = summarize(res)
    rows[name]["speedup_measured_verify"] = summarize(simulate(test_set, pol, cap, C, VERIFY))["speedup"]
summary_df = pd.DataFrame(rows).T
display(summary_df.round(3))

summary_df.to_csv(RES_DIR / "day3_summary_test_8b.csv")
fixed_df.to_csv(RES_DIR / "day3_fixed_k_full_8b.csv")
sweep_df.to_csv(RES_DIR / "day3_entropy_sweep_dev_8b.csv")
ax = summary_df.speedup.plot.barh(figsize=(7, 3)); ax.axvline(1, color="gray", lw=0.8)
ax.set_xlabel("speed-up (test)"); ax.set_title("Speed-up per policy (8B target / 1B draft)"); plt.tight_layout(); plt.show()

## Notes for the write-up
- Quote **simulated** speed-ups as primary and the formula as a cross-check; Day 4's real draft-and-verify loop validates them.
- `c` was measured with the 8B target layer-split over 2x T4 (sequential, with a GPU0 -> GPU1 hop per step) and the 1B draft on one GPU. On a single large GPU (A100/H100) the target step would be cheaper per layer-hop, so `c` would shift; say this explicitly.
- `c` must be the per-step decoding cost ratio (with KV cache), not prefill time. A wrong `c` changes every speed-up number.
- The gap `oracle - entropy` is the headroom for boundary-aware policies.
- Day 4 hook: add per-position arrays to each trace dict (e.g. `s["is_word_end"]`) and write policies with the same `(s, p, g) -> bool` signature.
- Compare against your 3B run: only the target changed, so differences in agreement rate, entropy, and `c` are attributable to the 8B target.

### Archive Experiment Artifacts

Bundles output files and evaluation tables for export:
- **Archive Creation**: Compresses `DATA_DIR` into `/kaggle/working/day2_day3_8b_artifacts.zip`.
- **Export**: Makes all traces, JSON profiles, and CSV summaries downloadable from Kaggle's working directory.

In [ ]:
# Bundle everything (outputs + day3 CSVs) into one zip in /kaggle/working so it shows up under the notebook's Output tab.
import shutil
zip_path = shutil.make_archive("/kaggle/working/day2_day3_8b_artifacts", "zip", DATA_DIR)
print("saved", zip_path)

### Reload Traces with Draft Predictions

Re-loads traces incorporating draft argmax token IDs:
- **Augmented Trace Loader**: Parses `draft_traces.json` and preserves `draft_argmax` arrays for each sentence.
- **Purpose**: Enables string decoding of drafted subwords for downstream linguistic boundary analysis.

In [ ]:
def load_traces(path):
    traces = []
    for r in json.load(open(path)):
        tgt = np.asarray(r["target_ids"]); drf = np.asarray(r["draft_argmax"])
        H = np.asarray(r["entropy"], dtype=float)
        match = (tgt == drf)
        traces.append(dict(id=r.get("id"), n=len(tgt), match=match, H=H, target_ids=tgt,
                           p_target=np.asarray(r.get("p_target", []), float), 
                           draft_argmax=drf, tokens=None)) # Added draft_argmax here
    return traces

traces = load_traces(TRACE_PATH)

### Linguistic Boundary Detection & Feature Tagging

Identifies lexical and subword orthographic boundaries in drafted Hindi tokens:
- **Resources**:
  - `tok`: AutoTokenizer for decoding draft token IDs.
  - `MORPH_DICT`: High-frequency Hindi grammatical and function morphemes.
  - `AKSHARA_REGEX`: Regex pattern identifying complete Devanagari syllabic units (aksharas).
- **Boundary Flags (`add_boundary_flags`)**:
  - `is_word_end`: True if next token initiates whitespace, newline, or punctuation.
  - `is_lookup_end`: True if accumulated tokens match an entry in `MORPH_DICT`.
  - `is_hybrid_end`: True if accumulated tokens form valid aksharas without dangling halants or combining signs.

In [ ]:
import re
from transformers import AutoTokenizer

# Load the tokenizer (Part B is CPU only, so just the tokenizer is fine)
tok = AutoTokenizer.from_pretrained(DRAFT if 'DRAFT' in locals() else "meta-llama/Llama-3.2-1B-Instruct")

# 1. Dictionary for Lookup (replace with your actual MorphTok dictionary if available)
# Using a dummy subset here for demonstration.
MORPH_DICT = {"है", "और", "में", "की", "से", "को", "का", "कि", "यह", "एक", "तो"} 

# 2. Devanagari Akshara Regex (from your design doc)
AKSHARA_REGEX = re.compile(r'(?:[\u0904-\u0939\u0958-\u0961](?:\u094D[\u0904-\u0939\u0958-\u0961])*[\u093E-\u094C\u0962\u0963]?[\u0901-\u0903]?)')

def add_boundary_flags(traces, tokenizer, morph_dict):
    print("Pre-calculating boundaries...")
    for s in traces:
        n = s["n"]
        drf = s["draft_argmax"]
        
        s["is_word_end"] = np.zeros(n, dtype=bool)
        s["is_lookup_end"] = np.zeros(n, dtype=bool)
        s["is_hybrid_end"] = np.zeros(n, dtype=bool)
        
        current_word_pieces = []
        
        for i in range(n):
            token_str = tokenizer.decode([drf[i]])
            
            # Word boundary: check if the NEXT token starts with space/newline or punctuation
            next_token_str = tokenizer.decode([drf[i+1]]) if i + 1 < n else " "
            if next_token_str.startswith(" ") or next_token_str.startswith("\n") or re.match(r'^[.,!?।]', next_token_str):
                s["is_word_end"][i] = True

            # Accumulate pieces for lookup/hybrid logic
            if token_str.startswith(" ") or token_str.startswith("\n"):
                current_word_pieces = [token_str.strip()]
            else:
                current_word_pieces.append(token_str)
                
            word_so_far = "".join(current_word_pieces).strip()
            
            # Policy 1: Lookup (Exact Match)
            if word_so_far in morph_dict:
                s["is_lookup_end"][i] = True
                s["is_hybrid_end"][i] = True
            else:
                # Policy 2: Hybrid Fallback (Regex validation of Akshara)
                # Check if the word strictly comprises complete aksharas without dangling halants/matras
                if len(word_so_far) > 0:
                    aksharas = AKSHARA_REGEX.findall(word_so_far)
                    if "".join(aksharas) == word_so_far:
                        s["is_hybrid_end"][i] = True

    return traces

traces = add_boundary_flags(traces, tok, MORPH_DICT)

### Linguistic Boundary-Aware Speculative Stopping Policies

Implements policies that halt drafting based on linguistic and orthographic units:
- **`WordBoundaryPolicy`**: Halts drafting when a complete word boundary (`is_word_end`) is encountered.
- **`LookupPolicy`**: Halts drafting when accumulated tokens match a recognized morpheme in `MORPH_DICT` (`is_lookup_end`).
- **`HybridPolicy`**: Halts drafting at complete akshara or lexical boundaries (`is_hybrid_end`), avoiding split syllables.

In [ ]:
class WordBoundaryPolicy:
    def __call__(self, s, p, g):
        if g == 0: return False
        return s["is_word_end"][p + g - 1]

class LookupPolicy:
    def __call__(self, s, p, g):
        if g == 0: return False
        return s["is_lookup_end"][p + g - 1]

class HybridPolicy:
    def __call__(self, s, p, g):
        if g == 0: return False
        return s["is_hybrid_end"][p + g - 1]

### Boundary Policy Evaluation & Entropy Correlation Analysis

Simulates boundary-aware policies and tests statistical redundancy with entropy:
- **Policy Simulation**: Evaluates `WordBoundaryPolicy`, `LookupPolicy`, and `HybridPolicy` on `test_set` and prints speedup, acceptance rate, and draft length.
- **Correlation Test**: Computes point-biserial correlation ($r_{\text{pb}}$) between token entropy $H$ and binary word boundaries (`is_word_end`) on `dev_set`.
- **Redundancy Diagnosis**: Assesses whether model uncertainty naturally aligns with word boundaries or if linguistic rules capture orthogonal stopping signals.

In [ ]:
from scipy.stats import pointbiserialr

# 1. Run the new policies
boundary_policies = {
    "word boundary": (WordBoundaryPolicy(), GAMMA_MAX),
    "morpheme lookup": (LookupPolicy(), GAMMA_MAX),
    "hybrid boundary": (HybridPolicy(), GAMMA_MAX)
}

print("--- Day 4 Boundary Policies ---")
for name, (pol, cap) in boundary_policies.items():
    res = simulate(test_set, pol, cap, C)
    m = summarize(res)
    print(f"{name:15s} | Speed-up: {m['speedup']:.3f}x | Acceptance: {m['acceptance_rate']:.3f} | Draft Len: {m['mean_draft_len']:.2f}")

# 2. Measure Correlation between Entropy and Word Boundaries (Redundancy Check)
all_H = []
all_is_word_end = []

for s in dev_set: # Measure on dev set
    all_H.extend(s["H"])
    all_is_word_end.extend(s["is_word_end"])

all_H = np.array(all_H)
all_is_word_end = np.array(all_is_word_end, dtype=int)

correlation, p_value = pointbiserialr(all_is_word_end, all_H)
print(f"\nCorrelation between Entropy and Word Boundary: {correlation:.3f} (p-value: {p_value:.3e})")
if correlation > 0.3:
    print("Insight: Boundaries correlate strongly with high entropy. The models naturally hesitate at word boundaries, meaning your entropy policy and boundary policy might be redundant.")
else:
    print("Insight: Low correlation. The hybrid boundary policy intercepts unique states that pure entropy thresholding misses.")

## Final Results Summary & Full Artifact Export

Consolidates every experiment's key metric into a single printed table, saves all results to CSV + JSON, and bundles everything into one downloadable zip.

In [ ]:
# =============================================================================
# FINAL RESULTS SUMMARY & ARTIFACT EXPORT
# Consolidates metrics from every experiment and saves all outputs to disk.
# =============================================================================

import json, shutil, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

# ── 1. Hardware / cost-ratio profile ────────────────────────────────────────
T = json.load(open(TIMING_PATH))
C = T["c"]
print(f"\n{'='*70}")
print("HARDWARE PROFILE")
print(f"{'='*70}")
print(f"  Target model    : {T['setup']['target']}")
print(f"  Draft  model    : {T['setup']['draft']}")
print(f"  dtype           : {T['setup']['dtype']}")
print(f"  GPUs            : {T['setup']['gpus']}")
print(f"  t_target (1 tok): {T['t_target']*1000:.2f} ms")
print(f"  t_draft  (1 tok): {T['t_draft']*1000:.2f} ms")
print(f"  Cost ratio c    : {C:.4f}")

# ── 2. Draft-trace statistics (over ALL traces) ──────────────────────────────
all_match   = np.concatenate([s["match"] for s in traces])
all_H       = np.concatenate([s["H"]     for s in traces])
all_p_tgt   = np.concatenate([s["p_target"] for s in traces])
n_sentences = len(traces)
n_tokens    = int(all_match.sum() + (~all_match).sum())   # == len(all_match)

print(f"\n{'='*70}")
print("DRAFT-TRACE STATISTICS  (all {n_sentences} sentences, {n_tokens} tokens)".format(**locals()))
print(f"{'='*70}")
print(f"  Overall token agreement (draft == target): {all_match.mean():.4f}")
print(f"  Mean entropy  (all positions)            : {all_H.mean():.4f} nats")
print(f"  Mean entropy  (matched positions)        : {all_H[all_match].mean():.4f} nats")
print(f"  Mean entropy  (mismatched positions)     : {all_H[~all_match].mean():.4f} nats")
print(f"  Mean p(target token)                     : {all_p_tgt.mean():.4f}")

# ── 3. Fixed-k results table ─────────────────────────────────────────────────
print(f"\n{'='*70}")
print("FIXED-k BASELINE  (all traces)")
print(f"{'='*70}")
fixed_rows = []
for k in K_RANGE:
    res = simulate(traces, FixedK(k), k, C)
    m   = summarize(res)
    fixed_rows.append({"k": k, **m,
                        "formula_speedup": formula_speedup(m["alpha_hat"], k, C),
                        "speedup_meas_verify": summarize(simulate(traces, FixedK(k), k, C, VERIFY))["speedup"]})
fixed_summary = pd.DataFrame(fixed_rows).set_index("k")
print(fixed_summary.round(4).to_string())

# ── 4. Entropy-policy grid (dev) – best config ───────────────────────────────
print(f"\n{'='*70}")
print("ENTROPY POLICY  (dev sweep → best config evaluated on test)")
print(f"{'='*70}")
print(f"  Best lambda : {best.lam:.3f}")
print(f"  Best beta   : {best.beta}")
ent_test = summarize(simulate(test_set, EntropyPolicy(best.lam, best.beta), GAMMA_MAX, C))
print(f"  Test speedup: {ent_test['speedup']:.4f}x")
print(f"  Acceptance  : {ent_test['acceptance_rate']:.4f}")
print(f"  Mean draft  : {ent_test['mean_draft_len']:.2f} tokens")

# ── 5. Oracle bounds ─────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("ORACLE UPPER BOUNDS  (test set)")
print(f"{'='*70}")
for cap, label in [(GAMMA_MAX, f"cap={GAMMA_MAX}"), (32, "cap=32 (uncapped)")]:
    r = summarize(simulate(test_set, Oracle(), cap, C))
    print(f"  Oracle ({label:16s}): speedup={r['speedup']:.4f}x  "
          f"tokens/verif={r['tokens_per_verif']:.2f}  "
          f"mean_draft={r['mean_draft_len']:.2f}")

# ── 6. Linguistic boundary policies ──────────────────────────────────────────
print(f"\n{'='*70}")
print("LINGUISTIC BOUNDARY POLICIES  (test set)")
print(f"{'='*70}")
boundary_results = {}
for name, (pol, cap) in boundary_policies.items():
    r = simulate(test_set, pol, cap, C)
    m = summarize(r)
    boundary_results[name] = m
    print(f"  {name:22s}: speedup={m['speedup']:.4f}x  "
          f"accept={m['acceptance_rate']:.4f}  "
          f"draft_len={m['mean_draft_len']:.2f}")

print(f"\n  Entropy–WordBoundary correlation (dev): r={correlation:.4f}  p={p_value:.2e}")

# ── 7. Full comparison table (test set) ──────────────────────────────────────
print(f"\n{'='*70}")
print("FULL POLICY COMPARISON  (test set)")
print(f"{'='*70}")

best_k_dev = max(K_RANGE, key=lambda k: summarize(simulate(dev_set, FixedK(k), k, C))["speedup"])
all_policies = {
    "no speculation"                                    : (NoDraft(),                              GAMMA_MAX),
    f"fixed k={best_k_dev} (best)"                     : (FixedK(best_k_dev),                     best_k_dev),
    f"entropy λ={best.lam:.2f} β={best.beta}"          : (EntropyPolicy(best.lam, best.beta),     GAMMA_MAX),
    "word boundary"                                     : (WordBoundaryPolicy(),                   GAMMA_MAX),
    "morpheme lookup"                                   : (LookupPolicy(),                         GAMMA_MAX),
    "hybrid akshara"                                    : (HybridPolicy(),                         GAMMA_MAX),
    f"oracle (cap {GAMMA_MAX})"                         : (Oracle(),                               GAMMA_MAX),
    "oracle (cap 32)"                                   : (Oracle(),                               32),
}
full_rows = {}
for name, (pol, cap) in all_policies.items():
    res = simulate(test_set, pol, cap, C)
    m   = summarize(res)
    m["speedup_meas_verify"] = summarize(simulate(test_set, pol, cap, C, VERIFY))["speedup"]
    full_rows[name] = m

full_df = pd.DataFrame(full_rows).T
print(full_df.round(4).to_string())

# ── 8. Save CSVs ─────────────────────────────────────────────────────────────
RES_DIR.mkdir(parents=True, exist_ok=True)
fixed_summary.round(6).to_csv(RES_DIR / "fixed_k_results.csv")
sweep_df.round(6).to_csv(RES_DIR / "entropy_sweep_dev.csv")
full_df.round(6).to_csv(RES_DIR / "full_policy_comparison_test.csv")
pd.DataFrame(boundary_results).T.round(6).to_csv(RES_DIR / "boundary_policy_test.csv")
print(f"\nCSVs saved to {RES_DIR}")

# ── 9. Master summary JSON ────────────────────────────────────────────────────
summary_dict = {
    "hardware": {
        "target": T["setup"]["target"],
        "draft":  T["setup"]["draft"],
        "dtype":  T["setup"]["dtype"],
        "gpus":   T["setup"]["gpus"],
        "t_target_ms": round(T["t_target"] * 1000, 4),
        "t_draft_ms":  round(T["t_draft"]  * 1000, 4),
        "cost_ratio_c": round(C, 6),
    },
    "dataset": {
        "name":       "openlanguagedata/flores_plus (eng_Latn → hin_Deva, devtest)",
        "n_sentences": n_sentences,
        "n_tokens":    n_tokens,
    },
    "draft_trace_stats": {
        "overall_agreement":    round(float(all_match.mean()), 6),
        "mean_entropy_nats":    round(float(all_H.mean()),     6),
        "mean_entropy_match":   round(float(all_H[all_match].mean()),   6),
        "mean_entropy_mismatch":round(float(all_H[~all_match].mean()),  6),
        "mean_p_target":        round(float(all_p_tgt.mean()), 6),
    },
    "fixed_k": fixed_summary.round(6).to_dict(orient="index"),
    "entropy_policy": {
        "best_lambda": float(best.lam),
        "best_beta":   float(best.beta),
        "test_metrics": {k: round(float(v), 6) for k, v in ent_test.items()},
    },
    "oracle": {
        f"cap_{GAMMA_MAX}": {k: round(float(v), 6)
                              for k, v in summarize(simulate(test_set, Oracle(), GAMMA_MAX, C)).items()},
        "cap_32":           {k: round(float(v), 6)
                              for k, v in summarize(simulate(test_set, Oracle(), 32, C)).items()},
    },
    "boundary_policies": {
        name: {k: round(float(v), 6) for k, v in m.items()}
        for name, m in boundary_results.items()
    },
    "entropy_boundary_correlation": {
        "r_pb":    round(float(correlation), 6),
        "p_value": float(p_value),
    },
    "full_policy_comparison_test": {
        name: {k: round(float(v), 6) for k, v in row.items()}
        for name, row in full_rows.items()
    },
}
json_path = RES_DIR / "master_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary_dict, f, indent=2, ensure_ascii=False)
print(f"Master JSON saved to {json_path}")

# ── 10. Summary figure ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# (a) Fixed-k: speedup + formula
ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(fixed_summary.index, fixed_summary["speedup"],              "o-",  label="simulated")
ax0.plot(fixed_summary.index, fixed_summary["formula_speedup"],      "s--", label="formula (i.i.d.)")
ax0.plot(fixed_summary.index, fixed_summary["speedup_meas_verify"],  "^:",  label="meas. verify")
ax0.axhline(1, color="gray", lw=0.8); ax0.legend(fontsize=7)
ax0.set(xlabel="k", ylabel="Speed-up", title="Fixed-k Speed-up")

# (b) Fixed-k: acceptance
ax1 = fig.add_subplot(gs[0, 1])
ax1.plot(fixed_summary.index, fixed_summary["acceptance_rate"], "o-", label="accepted/drafted")
ax1.plot(fixed_summary.index, fixed_summary["alpha_hat"],       "s--",label="alpha_hat")
ax1.set(xlabel="k", ylabel="Rate", title="Fixed-k Acceptance"); ax1.legend(fontsize=7)

# (c) Entropy sweep
ax2 = fig.add_subplot(gs[0, 2])
for beta, g in sweep_df.groupby("beta"):
    ax2.plot(g.lam, g.speedup, "o-", ms=3, label=f"β={beta}")
ax2.axhline(summarize(simulate(dev_set, FixedK(best_k_dev), best_k_dev, C))["speedup"],
            color="gray", ls="--", label=f"fixed k={best_k_dev}")
ax2.set(xlabel="λ", ylabel="Speed-up (dev)", title="Entropy Policy Sweep")
ax2.legend(fontsize=7)

# (d) Full policy comparison bar chart
ax3 = fig.add_subplot(gs[1, :])
speedups = {name: row["speedup"] for name, row in full_rows.items()}
colors   = ["#888888" if "no spec" in name
            else "#4c72b0" if "fixed" in name
            else "#dd8452" if "entropy" in name
            else "#55a868" if any(w in name for w in ["word", "morph", "hybrid"])
            else "#c44e52"  # oracle
            for name in speedups]
bars = ax3.barh(list(speedups.keys()), list(speedups.values()), color=colors)
ax3.axvline(1, color="gray", lw=0.9, ls="--")
for bar, val in zip(bars, speedups.values()):
    ax3.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
             f"{val:.3f}×", va="center", fontsize=8)
ax3.set(xlabel="Speed-up (test set)", title="Full Policy Comparison — 8B Target / 1B Draft (FLORES+ eng→hin)")
ax3.invert_yaxis()

plt.suptitle("Speculative Decoding: End-to-End Summary", fontsize=14, fontweight="bold")
fig_path = RES_DIR / "summary_figure.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Summary figure saved to {fig_path}")

# ── 11. Final archive ─────────────────────────────────────────────────────────
zip_path = shutil.make_archive("/kaggle/working/day2_day3_8b_final", "zip", DATA_DIR)
print(f"\nFinal archive: {zip_path}")
print(f"{'='*70}")
print("ALL DONE — artifacts summary:")
for p in sorted(RES_DIR.iterdir()):
    print(f"  {p.name:45s}  {p.stat().st_size / 1024:7.1f} KB")
